<a href="https://colab.research.google.com/github/Amrisha-90/careloop-clinic-analysis/blob/main/CareLoop_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
TASK 1.1 — Read the file

SyntaxError: invalid character '—' (U+2014) (2221159025.py, line 1)

In [ ]:
import csv
import json
import pandas as pd
import numpy as np

In [ ]:
with open("appointments.csv", "r", newline="") as file:
    reader = csv.DictReader(file)
    rows = list(reader)

print("Headers:", reader.fieldnames)
print("Total rows:", len(rows))

In [8]:
accepted = []
rejected = []

In [9]:
def parse_row(row):
    converted = row.copy()

    converted["age"] = int(row["age"])
    converted["fee_inr"] = int(row["fee_inr"])
    converted["prior_no_shows"] = int(row["prior_no_shows"])
    converted["distance_km"] = float(row["distance_km"])

    return converted

In [10]:
with open("rejected_rows.json", "w") as file:
    json.dump(rejected, file, indent=2)

In [11]:
Task 1.2

SyntaxError: invalid syntax (2117331241.py, line 1)

In [17]:
import pandas as pd

In [18]:
df = pd.read_csv("appointments.csv")

print("Original Shape:", df.shape)

Original Shape: (1020, 14)


In [19]:
print("Dataset Information:")
df.info()

Dataset Information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1020 entries, 0 to 1019
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   appointment_id   1020 non-null   object 
 1   patient_id       1020 non-null   object 
 2   clinic_id        1020 non-null   object 
 3   booked_on        1020 non-null   object 
 4   appt_date        1019 non-null   object 
 5   age              1008 non-null   float64
 6   gender           1020 non-null   object 
 7   distance_km      1000 non-null   float64
 8   fee_inr          1020 non-null   int64  
 9   booking_channel  1020 non-null   object 
 10  visit_type       1020 non-null   object 
 11  reminder_sent    1020 non-null   object 
 12  prior_no_shows   1020 non-null   int64  
 13  status           1016 non-null   object 
dtypes: float64(2), int64(2), object(10)
memory usage: 111.7+ KB


In [20]:
print("Missing Values:")
print(df.isnull().sum())

Missing Values:
appointment_id      0
patient_id          0
clinic_id           0
booked_on           0
appt_date           1
age                12
gender              0
distance_km        20
fee_inr             0
booking_channel     0
visit_type          0
reminder_sent       0
prior_no_shows      0
status              4
dtype: int64


In [21]:
print("Duplicate Rows:")
print(df.duplicated().sum())

Duplicate Rows:
20


In [22]:
df = df.drop_duplicates()

print("Rows after removing duplicates:", len(df))

Rows after removing duplicates: 1000


In [23]:
df["booking_channel"] = (
    df["booking_channel"].str.strip().str.title()
)

print("Rows after cleaning booking_channel:", len(df))

Rows after cleaning booking_channel: 1000


In [24]:
df["appt_date"] = pd.to_datetime(
    df["appt_date"], errors="coerce"
)

print("Rows after converting appt_date:", len(df))

Rows after converting appt_date: 1000


/tmp/ipykernel_5661/791493091.py:1: UserWarning: Parsing dates in %d-%m-%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df["appt_date"] = pd.to_datetime(


In [25]:
df["booked_on"] = pd.to_datetime(
    df["booked_on"]
)

print("Rows after converting booked_on:", len(df))

Rows after converting booked_on: 1000


/tmp/ipykernel_5661/1846422364.py:1: UserWarning: Parsing dates in %d-%m-%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df["booked_on"] = pd.to_datetime(


In [26]:
df = df.dropna(subset=["appt_date", "status"])

print("Rows after dropping missing values:", len(df))

Rows after dropping missing values: 990


In [27]:
print("Final Shape:", df.shape)

print("Remaining Missing Values:")
print(df.isnull().sum())

print("Remaining Duplicates:")
print(df.duplicated().sum())

Final Shape: (990, 14)
Remaining Missing Values:
appointment_id      0
patient_id          0
clinic_id           0
booked_on           0
appt_date           0
age                12
gender              0
distance_km        20
fee_inr             0
booking_channel     0
visit_type          0
reminder_sent       0
prior_no_shows      0
status              0
dtype: int64
Remaining Duplicates:
0


In [ ]:
Task 1.3

In [28]:
df.loc[(df["age"] < 1) | (df["age"] > 100), "age"] = np.nan

In [29]:
median_age = df["age"].median()

df["age"] = (
    df["age"]
    .fillna(median_age)
    .round()
    .astype(int)
)

print("Missing ages after filling:", df["age"].isnull().sum())

Missing ages after filling: 0


In [30]:
median_distance = df["distance_km"].median()

df["distance_km"] = df["distance_km"].fillna(median_distance)

print("Missing distances after filling:", df["distance_km"].isnull().sum())

Missing distances after filling: 0


In [31]:

Q1_distance = df["distance_km"].quantile(0.25)
Q3_distance = df["distance_km"].quantile(0.75)

IQR_distance = Q3_distance - Q1_distance

lower_distance = Q1_distance - 1.5 * IQR_distance
upper_distance = Q3_distance + 1.5 * IQR_distance

distance_outliers = (
    (df["distance_km"] < lower_distance) |
    (df["distance_km"] > upper_distance)
)

distance_capped = distance_outliers.sum()

df["distance_km"] = np.clip(
    df["distance_km"],
    lower_distance,
    upper_distance
)

print("Distance values capped:", distance_capped)

Distance values capped: 26


In [32]:
Q1_fee = df["fee_inr"].quantile(0.25)
Q3_fee = df["fee_inr"].quantile(0.75)

IQR_fee = Q3_fee - Q1_fee

lower_fee = Q1_fee - 1.5 * IQR_fee
upper_fee = Q3_fee + 1.5 * IQR_fee

fee_outliers = (
    (df["fee_inr"] < lower_fee) |
    (df["fee_inr"] > upper_fee)
)

fee_capped = fee_outliers.sum()

df["fee_inr"] = np.clip(
    df["fee_inr"],
    lower_fee,
    upper_fee
)

print("Fee values capped:", fee_capped)

Fee values capped: 3


In [33]:
df["distance_km"] = df["distance_km"].round(1)

df["fee_inr"] = df["fee_inr"].round().astype(int)

In [34]:
print("Final Dataset Shape:", df.shape)

print("\nAge Summary:")
print(df["age"].describe())

print("\nDistance Summary:")
print(df["distance_km"].describe())

print("\nFee Summary:")
print(df["fee_inr"].describe())

print("\nMissing Values:")
print(df.isnull().sum())

print("\nDistance values capped:", distance_capped)
print("Fee values capped:", fee_capped)

Final Dataset Shape: (990, 14)

Age Summary:
count    990.000000
mean      48.173737
std       16.655376
min       18.000000
25%       34.000000
50%       48.000000
75%       62.000000
max       78.000000
Name: age, dtype: float64

Distance Summary:
count    990.000000
mean       9.500909
std        6.160314
min        0.600000
25%        4.625000
50%        8.200000
75%       13.000000
max       25.600000
Name: distance_km, dtype: float64

Fee Summary:
count     990.000000
mean      635.714141
std       183.802489
min       300.000000
25%       500.000000
50%       650.000000
75%       787.500000
max      1219.000000
Name: fee_inr, dtype: float64

Missing Values:
appointment_id     0
patient_id         0
clinic_id          0
booked_on          0
appt_date          0
age                0
gender             0
distance_km        0
fee_inr            0
booking_channel    0
visit_type         0
reminder_sent      0
prior_no_shows     0
status             0
dtype: int64

Distance values cap

In [ ]:
Task - 1.4

In [35]:
df["lead_days"] = (df["appt_date"] - df["booked_on"]).dt.days

In [36]:
df["no_show"] = (df["status"] == "No-Show").astype(int)

print(df[["status", "no_show"]].head())

     status  no_show
0   No-Show        1
1  Attended        0
2  Attended        0
4  Attended        0
5   No-Show        1


In [37]:
X = df[["age", "distance_km", "fee_inr", "lead_days"]].to_numpy()

print("NumPy Array:")
print(X)

NumPy Array:
[[ 60.   18.  850.    1. ]
 [ 39.   12.5 350.   10. ]
 [ 63.    4.1 850.    6. ]
 ...
 [ 58.    3.2 850.   11. ]
 [ 59.   25.6 300.   13. ]
 [ 38.   10.8 550.   45. ]]


In [38]:
column_means = np.mean(X, axis=0)

print("Mean of each column:")
print(column_means)

Mean of each column:
[ 48.17373737   9.50090909 635.71414141  11.81919192]


In [39]:
mask = df["lead_days"] > 14

print("Number of rows with lead_days > 14:", mask.sum())

print("Share of rows with lead_days > 14:", mask.mean())

print("Percentage:", round(mask.mean() * 100, 1), "%")

Number of rows with lead_days > 14: 296
Share of rows with lead_days > 14: 0.298989898989899
Percentage: 29.9 %


In [40]:
distance = X[:, 1]

z_scores = (distance - np.mean(distance)) / np.std(distance)

print("Distance Z-scores:")
print(z_scores)

print("Number of rows with |z| > 2:", np.sum(np.abs(z_scores) > 2))

Distance Z-scores:
[ 1.38034946  0.48708663 -0.87716934 -0.3249705   1.54276088  0.34091634
  0.73070376 -0.55234649 -1.0558219  -0.73099906 -1.08830419 -0.3249705
  0.34091634 -0.38993506 -0.30872935 -1.38064475 -0.60106992  0.0160935
  0.04857578 -0.71475791 -1.26695676 -1.00709848 -0.79596363  1.2829026
 -1.26695676  0.2109872   1.72141345 -0.61731106 -0.48738192  1.33162603
  0.79566833  0.04857578 -0.99085733 -1.0558219  -0.47114078 -0.76348134
 -1.08830419  1.46155517 -0.37369392 -1.36440361 -1.13702762 -0.37369392
  2.17616544 -0.30872935  0.82815062  2.61467628  0.0160935   2.48474714
 -0.3249705  -0.2112825  -0.5198642  -0.40617621  0.8606329  -1.42936818
  1.00680318 -0.50362306  1.80261916 -0.79596363 -0.7472402  -0.19504136
  2.07871858 -0.45489963 -1.21823333 -0.87716934  0.71446262  0.19474606
  0.79566833 -0.16255907 -0.99085733 -0.82844591  0.30843406 -1.00709848
 -1.08830419 -0.19504136 -0.37369392 -0.87716934 -1.39688589  0.34091634
  0.34091634 -1.15326876 -1.0070984

In [41]:
print("Total rows:", len(df))

print("No-shows:", df["no_show"].sum())

print("Column means:", np.round(column_means, 2))

print("Lead days greater than 14:", mask.sum())

print("Lead days percentage:", round(mask.mean() * 100, 1), "%")

print("Distance Z-score greater than 2:", np.sum(np.abs(z_scores) > 2))

Total rows: 990
No-shows: 218
Column means: [ 48.17   9.5  635.71  11.82]
Lead days greater than 14: 296
Lead days percentage: 29.9 %
Distance Z-score greater than 2: 53


In [ ]:
Task - 1.5

In [42]:
import requests
import pandas as pd

In [44]:
clinics = pd.read_csv("clinics.csv")

print(clinics.head())
print("Total clinics:", len(clinics))

  clinic_id             clinic_name    city    tier  latitude  longitude
0       C01   Kothrud Family Clinic    Pune   Metro   18.5074    73.8077
1       C02    Hadapsar Care Centre    Pune   Metro   18.5089    73.9260
2       C03  Sitabuldi Health Point  Nagpur  Tier-2   21.1458    79.0882
3       C04    Malviya Nagar Clinic  Jaipur  Tier-2   26.8553    75.8150
4       C05   Vaishali Nagar Clinic  Jaipur  Tier-2   26.9124    75.7873
Total clinics: 8


In [45]:
url = "https://api.open-meteo.com/v1/forecast"

In [53]:
temperatures = []

for index, clinic in clinics.iterrows():

    params = {
        "latitude": clinic["latitude"],
        "longitude": clinic["longitude"],
        "hourly": "temperature_2m",
        "forecast_days": 1
    }

    try:
        response = requests.get(url, params=params, timeout=10)
        response.raise_for_status()

        data = response.json()
        temp_c = data["hourly"]["temperature_2m"][0]

    except Exception as e:
        print(clinic["clinic_name"], "Failed:", e)
        temp_c = None

    temperatures.append(temp_c)

In [54]:
clinics["temp_c"] = temperatures

print(clinics)

  clinic_id               clinic_name     city    tier  latitude  longitude  \
0       C01     Kothrud Family Clinic     Pune   Metro   18.5074    73.8077   
1       C02      Hadapsar Care Centre     Pune   Metro   18.5089    73.9260   
2       C03    Sitabuldi Health Point   Nagpur  Tier-2   21.1458    79.0882   
3       C04      Malviya Nagar Clinic   Jaipur  Tier-2   26.8553    75.8150   
4       C05     Vaishali Nagar Clinic   Jaipur  Tier-2   26.9124    75.7873   
5       C06  Edappally Medical Centre    Kochi  Tier-2   10.0261    76.3085   
6       C07        Vijay Nagar Clinic   Indore  Tier-2   22.7533    75.8937   
7       C08    Gomti Nagar Care Point  Lucknow  Tier-2   26.8467    80.9462   

   temp_c  
0    14.4  
1    15.7  
2    22.3  
3    15.3  
4    15.3  
5    24.2  
6    15.2  
7    17.0  


In [55]:
clinics.to_csv("clinics_enriched.csv", index=False)

print("clinics_enriched.csv saved successfully!")

clinics_enriched.csv saved successfully!


In [56]:
print(clinics[["clinic_name", "temp_c"]])

                clinic_name  temp_c
0     Kothrud Family Clinic    14.4
1      Hadapsar Care Centre    15.7
2    Sitabuldi Health Point    22.3
3      Malviya Nagar Clinic    15.3
4     Vaishali Nagar Clinic    15.3
5  Edappally Medical Centre    24.2
6        Vijay Nagar Clinic    15.2
7    Gomti Nagar Care Point    17.0


In [59]:
clinics["temp_c"] = temperatures

print(clinics)

  clinic_id               clinic_name     city    tier  latitude  longitude  \
0       C01     Kothrud Family Clinic     Pune   Metro   18.5074    73.8077   
1       C02      Hadapsar Care Centre     Pune   Metro   18.5089    73.9260   
2       C03    Sitabuldi Health Point   Nagpur  Tier-2   21.1458    79.0882   
3       C04      Malviya Nagar Clinic   Jaipur  Tier-2   26.8553    75.8150   
4       C05     Vaishali Nagar Clinic   Jaipur  Tier-2   26.9124    75.7873   
5       C06  Edappally Medical Centre    Kochi  Tier-2   10.0261    76.3085   
6       C07        Vijay Nagar Clinic   Indore  Tier-2   22.7533    75.8937   
7       C08    Gomti Nagar Care Point  Lucknow  Tier-2   26.8467    80.9462   

   temp_c  
0    14.4  
1    15.7  
2    22.3  
3    15.3  
4    15.3  
5    24.2  
6    15.2  
7    17.0  


In [50]:
print("Available columns:")
print(clinics.columns.tolist())

Available columns:
['clinic_id', 'clinic_name', 'city', 'tier', 'latitude', 'longitude', 'temp_c']


In [60]:
print("Total clinics:", len(clinics))

print("\nTemperature values:")
print(clinics[["clinic_name", "temp_c"]])

print("\nMissing temperatures:")
print(clinics["temp_c"].isnull().sum())

Total clinics: 8

Temperature values:
                clinic_name  temp_c
0     Kothrud Family Clinic    14.4
1      Hadapsar Care Centre    15.7
2    Sitabuldi Health Point    22.3
3      Malviya Nagar Clinic    15.3
4     Vaishali Nagar Clinic    15.3
5  Edappally Medical Centre    24.2
6        Vijay Nagar Clinic    15.2
7    Gomti Nagar Care Point    17.0

Missing temperatures:
0


In [61]:
print("TASK 1.5 COMPLETED")

print("\nTotal clinics:", len(clinics))

print("\nClinic details:")
print(clinics[["clinic_id", "clinic_name", "city", "temp_c"]])

print("\nMissing temperature values:")
print(clinics["temp_c"].isnull().sum())

print("\nSaved file: clinics_enriched.csv")

TASK 1.5 COMPLETED

Total clinics: 8

Clinic details:
  clinic_id               clinic_name     city  temp_c
0       C01     Kothrud Family Clinic     Pune    14.4
1       C02      Hadapsar Care Centre     Pune    15.7
2       C03    Sitabuldi Health Point   Nagpur    22.3
3       C04      Malviya Nagar Clinic   Jaipur    15.3
4       C05     Vaishali Nagar Clinic   Jaipur    15.3
5       C06  Edappally Medical Centre    Kochi    24.2
6       C07        Vijay Nagar Clinic   Indore    15.2
7       C08    Gomti Nagar Care Point  Lucknow    17.0

Missing temperature values:
0

Saved file: clinics_enriched.csv


In [ ]:
Task - 1.6

In [62]:
df.to_csv("appointments_clean.csv", index=False)

print("appointments_clean.csv saved successfully!")
print("Total rows:", len(df))
print("Total columns:", len(df.columns))

appointments_clean.csv saved successfully!
Total rows: 990
Total columns: 16


In [63]:
sql_script = """
CREATE TABLE IF NOT EXISTS appointments (
    appointment_id VARCHAR(50),
    patient_id VARCHAR(50),
    clinic_id VARCHAR(50),
    appt_date DATE,
    booked_on DATE,
    age INTEGER,
    gender VARCHAR(20),
    booking_channel VARCHAR(50),
    fee_inr INTEGER,
    prior_no_shows INTEGER,
    distance_km FLOAT,
    status VARCHAR(30),
    lead_days INTEGER,
    no_show INTEGER
);
"""

with open("load_data.sql", "w") as file:
    file.write(sql_script)

print("load_data.sql created successfully!")

load_data.sql created successfully!


In [64]:
import os

print(os.listdir())

['.config', 'clinics.csv', 'rejected_rows.json', 'clinics_enriched.csv', 'appointments_clean.csv', 'appointments.csv', 'load_data.sql', 'sample_data']


In [65]:
from google.colab import files

files.download("appointments_clean.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [66]:
files.download("clinics_enriched.csv")
files.download("load_data.sql")
files.download("rejected_rows.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>